# Notebook 07: Panel Regressions & Panel VAR
This notebook implements the core econometric regressions: disaggregated policy pressure regressions (Labor vs. Community), ESEA waiver interaction models ($H_7$), and a Helmert-transformed GMM Panel VAR to Granger-test feedback loops while correcting for Nickell bias.

### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*

## 1. Library Imports & Data Realignment
Load state-year panel data and merge disaggregated weights from the ESSA coding sheet.

In [1]:
import pandas as pd
import numpy as np

# Load state-year panel data (which contains real political and waiver variables)
df = pd.read_csv('../data/processed/state_year_panel.csv')
df_essa = pd.read_csv('../data/raw/essa_plan_coding_sheet.csv')

# Merge disaggregated weights
df = df.merge(df_essa[['state', 'academic_achievement_weight', 'academic_growth_weight']], on='state', how='left')

# Verify we have has_waiver and political controls loaded directly from panel
print('Data aligned. Sample size:', df.shape)
print('Waiver states:', list(df[df['has_waiver'] == 1]['state'].unique()[:5]))
print('Trifecta years:', df['trifecta'].sum())


Data aligned. Sample size: (765, 54)
Waiver states: ['AK', 'AL', 'AR', 'AZ', 'CO']
Trifecta years: 377


## 2. Construct Disaggregated Policy Pressure Indices
We construct separate indices for:
*   **Community-directed pressure** (parent-salient): uses pre-ESSA policy intensity, and post-ESSA academic achievement weight.
*   **Labor-directed pressure** (teacher-salient): uses pre-ESSA policy intensity, and post-ESSA academic growth weight.

In [2]:
df['raw_comm'] = df['raw_intensity']
df.loc[df['year'] >= 2018, 'raw_comm'] = df.loc[df['year'] >= 2018, 'academic_achievement_weight'] / 100.0

df['raw_labor'] = df['raw_intensity']
df.loc[df['year'] >= 2018, 'raw_labor'] = df.loc[df['year'] >= 2018, 'academic_growth_weight'] / 100.0

# Standardize within-era
df['policy_community'] = 0.0
df['policy_labor'] = 0.0

for mask in [df['year'] <= 2017, df['year'] >= 2018]:
    for col, target in [('raw_comm', 'policy_community'), ('raw_labor', 'policy_labor')]:
        m = df.loc[mask, col].mean()
        s = df.loc[mask, col].std()
        if pd.isna(s) or s == 0: s = 1.0
        df.loc[mask, target] = (df.loc[mask, col] - m) / s

print('Disaggregated policy indices constructed successfully.')

Disaggregated policy indices constructed successfully.


C:\Users\admir\AppData\Local\Temp\ipykernel_33228\587494806.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.35  0.35  0.35  0.35  0.35  0.35  0.35  0.4   0.4   0.4   0.4   0.4
 0.4   0.4   0.35  0.35  0.35  0.35  0.35  0.35  0.35  0.3   0.3   0.3
 0.3   0.3   0.3   0.3   0.4   0.4   0.4   0.4   0.4   0.4   0.4   0.3
 0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3
 0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.35  0.35  0.35  0.35
 0.35  0.35  0.35  0.333 0.333 0.333 0.333 0.333 0.333 0.333 0.3   0.3
 0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3
 0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3
 0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.4   0.4   0.4
 0.4   0.4   0.4   0.4   0.3   0.3   0.3   0.3   0.3   0.3   0.3   0.4
 0.4   0.4   0.4   0.4   0.4   0.4   0.4   0.4   0.4   0.4   0.4   0.4
 0.4   0.3   0.3   0.3 

## 3. Disaggregated Backlash Regressions
We estimate the impact of our disaggregated policy indices on backlash scores using OLS with state and year fixed effects and cluster-robust standard errors.

In [3]:
# Create lagged variables within each state
df['policy_lag1'] = df.groupby('state')['policy_intensity'].shift(1)
df['policy_comm_lag1'] = df.groupby('state')['policy_community'].shift(1)
df['policy_labor_lag1'] = df.groupby('state')['policy_labor'].shift(1)

from linearmodels.panel import PanelOLS

# Drop NaNs to prevent panel regression index alignment issues
cols_p = ['state', 'year', 'backlash', 'policy_lag1', 'policy_comm_lag1', 'policy_labor_lag1', 'gov_party_rep', 'trifecta', 'election_year']
df_p = df[cols_p].dropna().set_index(['state', 'year'])

# Model 1: Baseline Policy on Backlash
fit1 = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print('=== Baseline Policy on Backlash (PanelOLS) ===')
print(fit1.summary.tables[1])

# Model 2: Community-Directed Policy on Backlash
fit2 = PanelOLS.from_formula(
    'backlash ~ policy_comm_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== Community-Directed Policy on Backlash (PanelOLS) ===')
print(fit2.summary.tables[1])

# Model 3: Labor-Directed Policy on Backlash
fit3 = PanelOLS.from_formula(
    'backlash ~ policy_labor_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== Labor-Directed Policy on Backlash (PanelOLS) ===')
print(fit3.summary.tables[1])


=== Baseline Policy on Backlash (PanelOLS) ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1      -0.1491     0.1090    -1.3683     0.1717     -0.3632      0.0649
gov_party_rep    -0.2124     0.1244    -1.7082     0.0881     -0.4566      0.0318
trifecta         -1.0044     0.5081    -1.9769     0.0485     -2.0020     -0.0067
election_year     0.0494     0.0363     1.3596     0.1744     -0.0219      0.1207

=== Community-Directed Policy on Backlash (PanelOLS) ===
                                Parameter Estimates                                 
                  Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------
policy_comm_lag1     0.0089     0.0520     0.1706     0.8646     -

## 4. ESEA Waiver Dampening Interactions ($H_7$)
We test if ESEA waiver status (safety valve) buffers the policy-backlash link (H7a) and the backlash-correction link (H7b).

In [4]:
# Lagged variables for H7 interaction checks
df['backlash_lag1'] = df.groupby('state')['backlash'].shift(1)
df['delta_policy'] = df.groupby('state')['policy_intensity'].diff()

# Explicitly construct interaction terms to prevent index/parsing issues
df['policy_x_waiver'] = df['policy_lag1'] * df['has_waiver']
df['backlash_x_waiver'] = df['backlash_lag1'] * df['has_waiver']

cols_h7 = ['state', 'year', 'backlash', 'policy_lag1', 'has_waiver', 'policy_x_waiver', 'gov_party_rep', 'trifecta', 'delta_policy', 'backlash_lag1', 'backlash_x_waiver']
df_h7 = df[cols_h7].dropna().set_index(['state', 'year'])

# H7a: Backlash Dampening
fit_h7a = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + has_waiver + policy_x_waiver + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_h7
).fit(cov_type='clustered', cluster_entity=True)
print('=== H7a: Backlash Dampening (ESEA Waiver interaction) ===')
print(fit_h7a.summary.tables[1])

# H7b: Correction Dampening
fit_h7b = PanelOLS.from_formula(
    'delta_policy ~ backlash_lag1 + has_waiver + backlash_x_waiver + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_h7
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== H7b: Correction Dampening (ESEA Waiver interaction) ===')
print(fit_h7b.summary.tables[1])


=== H7a: Backlash Dampening (ESEA Waiver interaction) ===
                                Parameter Estimates                                
                 Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------
policy_lag1        -0.1773     0.1073    -1.6521     0.0990     -0.3881      0.0334
has_waiver          0.0884     0.1714     0.5156     0.6063     -0.2482      0.4250
policy_x_waiver     0.0688     0.0952     0.7224     0.4703     -0.1181      0.2557
gov_party_rep      -0.2254     0.1198    -1.8819     0.0603     -0.4607      0.0098
trifecta           -1.0005     0.4947    -2.0227     0.0435     -1.9719     -0.0292

=== H7b: Correction Dampening (ESEA Waiver interaction) ===
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------

## 5. Helmert-Transformed Panel VAR
To eliminate Nickell bias ($1/T$) in our panel, we apply the forward orthogonal deviations (Helmert transformation) to our variables, then estimate the vector autoregression.

In [5]:
def helmert_transform(df, cols):
    df_transformed = []
    for state, group in df.groupby('state'):
        group = group.sort_values('year').copy()
        T = len(group)
        for t in range(T - 1):
            row = group.iloc[t].copy()
            for col in cols:
                forward_vals = group.iloc[t+1:][col].values
                mean_forward = np.mean(forward_vals)
                scale = np.sqrt((T - t - 1) / (T - t))
                row[col] = scale * (group.iloc[t][col] - mean_forward)
            df_transformed.append(row)
    return pd.DataFrame(df_transformed)

# Estimating a 2-variable Panel VAR using IV/2SLS (Arellano-Bover GMM-style) to prevent boundary leaks & Nickell bias
# 1. Transform variables for education panel
df_var_input = df[['state', 'year', 'policy_intensity', 'backlash']].dropna().copy()
df_helmert = helmert_transform(df_var_input, ['policy_intensity', 'backlash'])

# 2. Create lagged variables within each state before dropping NaNs (Task 6 Boundary Leak Fix)
df_helmert['L1_policy_intensity'] = df_helmert.groupby('state')['policy_intensity'].shift(1)
df_helmert['L1_backlash'] = df_helmert.groupby('state')['backlash'].shift(1)
df_helmert['L2_policy_intensity'] = df_helmert.groupby('state')['policy_intensity'].shift(2)
df_helmert['L2_backlash'] = df_helmert.groupby('state')['backlash'].shift(2)

df_var_clean = df_helmert.dropna(subset=['L1_policy_intensity', 'L1_backlash', 'L2_policy_intensity', 'L2_backlash']).copy()

# 3. Fit equation-by-equation IV2SLS, using L2/L3 levels as instruments
from linearmodels.iv import IV2SLS

print('=== Equation 1: policy_intensity ~ L1_policy_intensity + L1_backlash (Instrumented by L2 levels) ===')
res_eq1 = IV2SLS.from_formula(
    'policy_intensity ~ 1 + [L1_policy_intensity + L1_backlash ~ L2_policy_intensity + L2_backlash]',
    data=df_var_clean
).fit(cov_type='clustered', clusters=df_var_clean['state'])
print(res_eq1.summary.tables[1])

print('\n=== Equation 2: backlash ~ L1_policy_intensity + L1_backlash (Instrumented by L2 levels) ===')
res_eq2 = IV2SLS.from_formula(
    'backlash ~ 1 + [L1_policy_intensity + L1_backlash ~ L2_policy_intensity + L2_backlash]',
    data=df_var_clean
).fit(cov_type='clustered', clusters=df_var_clean['state'])
print(res_eq2.summary.tables[1])


=== Equation 1: policy_intensity ~ L1_policy_intensity + L1_backlash (Instrumented by L2 levels) ===
                                  Parameter Estimates                                  
                     Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------------
Intercept               0.0868     0.0137     6.3299     0.0000      0.0599      0.1136
L1_policy_intensity     0.2236     0.0684     3.2693     0.0011      0.0895      0.3576
L1_backlash            -0.0110     0.0266    -0.4112     0.6810     -0.0632      0.0413

=== Equation 2: backlash ~ L1_policy_intensity + L1_backlash (Instrumented by L2 levels) ===
                                  Parameter Estimates                                  
                     Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------------
Intercept    

## 6. Save the Regressions Outputs
We save the updated dataframe back to disk to verify the columns exist for visualization.

In [6]:
df.to_csv('../data/processed/state_year_panel.csv', index=False)
print('Panel dataset updated with regression columns.')

Panel dataset updated with regression columns.
